# Univariate and Multivariate Analysis
### Seaborn Diamonds Dataset
---
**Dataset:** `diamonds` from seaborn  
**Variables chosen:**
- **Univariate Analysis** → `price` (continuous numeric variable)
- **Multivariate Analysis** → `carat` vs `price`, colored by `cut` (numeric + categorical)


## 1. Import Libraries

In [ ]:
# Core data manipulation and numerical computation
import numpy as np
import pandas as pd

# Seaborn for the diamonds dataset and statistical plotting
import seaborn as sns

# Matplotlib for plot customization and rendering
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

# SciPy for statistical tests (skewness, normality test)
from scipy import stats

# Display settings
pd.set_option('display.float_format', lambda x: f'{x:,.2f}')
plt.rcParams['figure.dpi'] = 120
print("All libraries imported successfully.")


## 2. Load the Diamonds Dataset

In [ ]:
# Load the built-in seaborn diamonds dataset
diamonds = sns.load_dataset('diamonds')

# Quick overview
print(f"Shape: {diamonds.shape[0]:,} rows x {diamonds.shape[1]} columns\n")
print(diamonds.dtypes)
print("\nFirst 5 rows:")
diamonds.head()


In [ ]:
# Check for missing values
print("Missing values per column:")
print(diamonds.isnull().sum())
print(f"\nTotal missing: {diamonds.isnull().sum().sum()}")


## 3. Univariate Analysis — `price`

**Variable chosen: `price`** (USD)  
Univariate analysis examines a single variable in isolation to understand its  
distribution, central tendency, spread, and shape.


In [ ]:
# ──────────────────────────────────────────────────
# UNIVARIATE ANALYSIS: price
# We chose 'price' because it is the key outcome variable
# in the diamonds dataset and typically has an interesting
# right-skewed distribution worth exploring.
# ──────────────────────────────────────────────────

price = diamonds['price']

# ── Descriptive statistics ──
print("=" * 45)
print("DESCRIPTIVE STATISTICS — price (USD)")
print("=" * 45)
print(f"  Count      : {price.count():>10,.0f}")
print(f"  Mean       : ${price.mean():>10,.2f}")
print(f"  Median     : ${price.median():>10,.2f}")
print(f"  Std Dev    : ${price.std():>10,.2f}")
print(f"  Min        : ${price.min():>10,.2f}")
print(f"  25th pct   : ${price.quantile(0.25):>10,.2f}")
print(f"  75th pct   : ${price.quantile(0.75):>10,.2f}")
print(f"  Max        : ${price.max():>10,.2f}")
print(f"  Skewness   : {stats.skew(price):>10.4f}")
print(f"  Kurtosis   : {stats.kurtosis(price):>10.4f}")

# ── Normality test (D'Agostino-Pearson on a 2000-sample subset) ──
sample = price.sample(2000, random_state=42)
stat, p = stats.normaltest(sample)
print(f"\nNormality test (p-value): {p:.6f}")
print("  → Distribution is NOT normal." if p < 0.05 else "  → Distribution appears normal.")


In [ ]:
# ── Visualisation: 4-panel univariate dashboard ──
fig, axes = plt.subplots(2, 2, figsize=(13, 9))
fig.suptitle("Univariate Analysis — Diamond Price (USD)", fontsize=15, fontweight='bold', y=1.01)

# Panel 1: Histogram with KDE
ax1 = axes[0, 0]
sns.histplot(price, bins=60, kde=True, color='steelblue', ax=ax1)
ax1.axvline(price.mean(),   color='red',    linestyle='--', linewidth=1.4, label=f'Mean  ${price.mean():,.0f}')
ax1.axvline(price.median(), color='orange', linestyle='--', linewidth=1.4, label=f'Median ${price.median():,.0f}')
ax1.set_title("Histogram with KDE")
ax1.set_xlabel("Price (USD)")
ax1.set_ylabel("Frequency")
ax1.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))
ax1.legend(fontsize=9)

# Panel 2: Box plot
ax2 = axes[0, 1]
sns.boxplot(y=price, color='skyblue', ax=ax2)
ax2.set_title("Box Plot")
ax2.set_ylabel("Price (USD)")
ax2.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))

# Panel 3: Violin plot
ax3 = axes[1, 0]
sns.violinplot(y=price, color='lightcoral', inner='quartile', ax=ax3)
ax3.set_title("Violin Plot")
ax3.set_ylabel("Price (USD)")
ax3.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))

# Panel 4: QQ-plot (tests normality visually)
ax4 = axes[1, 1]
stats.probplot(price, dist='norm', plot=ax4)
ax4.set_title("Q-Q Plot (Normality Check)")

plt.tight_layout()
plt.savefig('univariate_price.png', bbox_inches='tight')
plt.show()
print("\nInterpretation:")
print("  • The histogram and KDE show a strong RIGHT SKEW — most diamonds are")
print("    priced under $5,000, but a long tail extends to ~$19,000.")
print("  • Mean ($3,933) >> Median ($2,401), confirming the positive skew.")
print("  • The Q-Q plot deviates at both tails, confirming non-normality.")
print("  • The box plot reveals many high-price outliers above the upper fence.")


## 4. Multivariate Analysis — `carat` vs `price` by `cut`

**Variables chosen:**  
- `carat` (continuous numeric — diamond weight)  
- `price` (continuous numeric — USD)  
- `cut` (ordinal categorical — Fair / Good / Very Good / Premium / Ideal)  

Multivariate analysis examines relationships between two or more variables simultaneously.


In [ ]:
# ──────────────────────────────────────────────────
# MULTIVARIATE ANALYSIS: carat vs price, grouped by cut
#
# We chose carat and price because carat (weight) is the
# strongest predictor of price in the diamond industry.
# Adding 'cut' as a third variable lets us see whether
# cut quality modifies the carat–price relationship.
# ──────────────────────────────────────────────────

# ── Pearson correlation: carat vs price ──
r, p = stats.pearsonr(diamonds['carat'], diamonds['price'])
print(f"Pearson r (carat vs price): {r:.4f}  |  p-value: {p:.2e}")

# ── Correlation matrix for numeric columns ──
print("\nCorrelation matrix (numeric features):")
corr = diamonds.select_dtypes(include='number').corr()
print(corr.to_string())


In [ ]:
# ── Visualisation: 4-panel multivariate dashboard ──
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle("Multivariate Analysis — Carat vs Price by Cut", fontsize=15, fontweight='bold', y=1.01)

cut_order   = ['Fair', 'Good', 'Very Good', 'Premium', 'Ideal']
palette     = sns.color_palette("Set2", n_colors=5)

# ── Panel 1: Scatter plot (carat vs price, hued by cut) ──
ax1 = axes[0, 0]
# Sample 5,000 points for clarity (full 54k is too dense)
sample_df = diamonds.sample(5000, random_state=42)
sns.scatterplot(
    data=sample_df, x='carat', y='price',
    hue='cut', hue_order=cut_order, palette=palette,
    alpha=0.55, s=18, ax=ax1
)
ax1.set_title("Scatter: Carat vs Price (n=5,000 sample)")
ax1.set_xlabel("Carat")
ax1.set_ylabel("Price (USD)")
ax1.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))
ax1.legend(title='Cut', fontsize=8, title_fontsize=9)

# ── Panel 2: Regression lines per cut ──
ax2 = axes[0, 1]
for cut, color in zip(cut_order, palette):
    subset = diamonds[diamonds['cut'] == cut]
    sns.regplot(
        data=subset, x='carat', y='price',
        scatter=False, label=cut, color=color, ax=ax2,
        line_kws={'linewidth': 1.8}
    )
ax2.set_title("Regression Lines by Cut Quality")
ax2.set_xlabel("Carat")
ax2.set_ylabel("Price (USD)")
ax2.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))
ax2.legend(title='Cut', fontsize=8, title_fontsize=9)

# ── Panel 3: Box plot — price distribution per cut ──
ax3 = axes[1, 0]
sns.boxplot(data=diamonds, x='cut', y='price', order=cut_order, palette=palette, ax=ax3)
ax3.set_title("Price Distribution by Cut Quality")
ax3.set_xlabel("Cut")
ax3.set_ylabel("Price (USD)")
ax3.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))

# ── Panel 4: Heatmap — correlation matrix ──
ax4 = axes[1, 1]
sns.heatmap(
    corr, annot=True, fmt='.2f', cmap='coolwarm',
    center=0, square=True, linewidths=0.5, ax=ax4,
    annot_kws={'size': 9}
)
ax4.set_title("Correlation Heatmap (Numeric Features)")

plt.tight_layout()
plt.savefig('multivariate_carat_price_cut.png', bbox_inches='tight')
plt.show()
print("\nInterpretation:")
print("  • Strong positive correlation (r ≈ 0.92) between carat and price.")
print("  • 'Fair' cut diamonds are paradoxically priced higher per carat range")
print("    than 'Ideal' cuts — likely a confound with larger carat sizes.")
print("  • The regression lines show similar slopes across cuts, suggesting")
print("    carat weight dominates price more than cut quality does.")
print("  • x, y, z (physical dimensions) also highly correlate with price")
print("    since they directly relate to carat weight.")


## 5. Summary of Findings

| Analysis | Variable(s) | Key Finding |
|---|---|---|
| **Univariate** | `price` | Right-skewed; median \$2,401, mean \$3,933; many high-value outliers |
| **Multivariate** | `carat` × `price` × `cut` | Strong positive correlation (r≈0.92); carat weight is the dominant price driver regardless of cut quality |

**Takeaway:** Diamond pricing is overwhelmingly driven by carat weight. Cut quality has a secondary effect, and its relationship with price is partly obscured by the tendency for lower-quality cuts to appear on heavier stones.
